In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta


In [62]:
categories=pd.read_csv('../datasets/domens_categories.csv')
ranks=pd.read_csv('../datasets/ranked_top1000000.csv')
secondary_prices=pd.read_csv('../datasets/secondary_prices.csv')

Датасет с целевой переменной

In [63]:
ton_domens=pd.read_csv('../raw_datasets/ton_domains_history.csv').drop(['tx_hash','sale_type'],axis=1)

убираем .ton

In [64]:
ton_domens['domain_name']=ton_domens['domain_name'].map(lambda x: x.split('.')[0])

Так как это список продаж, один и тот же домен может встречаться несколько раз. Избавляемся от дубликатов, а актуальную цену берём по последней продаже 

In [65]:
ton_domens['tx_time'] = pd.to_datetime(ton_domens['tx_time'], unit='s')
ton_domens.sort_values(by='tx_time', inplace=True)
ton_domens = ton_domens.groupby(by='domain_name', as_index=False).agg(ton_price=('price_ton', 'last'))
ton_domens = ton_domens.rename(columns={'domain_name': 'domain'})

Собираем итоговый датасет

In [66]:
ton_dataset = ton_domens.merge(categories, on='domain', how='left')
len(ton_dataset[ton_dataset['category'].notna()])

3121

Всего 3121 доменов размеченных по категориям

In [67]:
ton_dataset=ton_dataset.merge(ranks, on='domain', how='left')

In [68]:
secondary_prices.rename(columns={'domain_name':'domain'}, inplace=True)
main_dataset = secondary_prices.merge(ton_dataset, on='domain', how='left')
main_dataset

,domain,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,sol_reg_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank
0,rilxxlir,17.22,NaN,NaN,NaN,19.387756,NaN,NaN,NaN,NaN,NaN,NaN
1,hansberry,8.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lalegend,6.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,peterwagner,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,stateland,1.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3490611,gangster-all-star,1.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490612,chain-feeds,6.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490613,68435,21.78,1.0,38.41376,38.41376,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490614,prismai-deployer,2.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Добавим признаки has_digit, has_letter, has_huphen и length

In [69]:
main_dataset["has_digit"] = main_dataset["domain"].str.contains(r"\\d").astype(int)
main_dataset["has_letter"] = main_dataset["domain"].str.contains(r"[A-Za-z]").astype(int)
main_dataset["has_huphen"] = main_dataset["domain"].str.contains("-").astype(int)  # опечатался, хотел has_hyphen написать но пофиг уже
main_dataset["length"] = main_dataset["domain"].str.len()
main_dataset

,domain,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,sol_reg_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank,has_digit,has_letter,has_huphen,length
0,rilxxlir,17.22,NaN,NaN,NaN,19.387756,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,8.0
1,hansberry,8.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,9.0
2,lalegend,6.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,8.0
3,peterwagner,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,11.0
4,stateland,1.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3490611,gangster-all-star,1.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1,17.0
3490612,chain-feeds,6.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1,11.0
3490613,68435,21.78,1.0,38.41376,38.41376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,5.0
3490614,prismai-deployer,2.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1,16.0


In [70]:
dataset_with_ton = main_dataset[main_dataset["ton_price"].notna()]
dataset_without_ton = main_dataset[main_dataset["ton_price"].isna()]
dataset_with_ton.to_csv("../datasets/dataset_main.csv", index=False)
dataset_without_ton.to_csv("../datasets/dataset_without_target_value.csv", index=False)
dataset_with_ton

,domain,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,sol_reg_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank,has_digit,has_letter,has_huphen,length
59,las-vegas-sands,10.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,0,1,1,15.0
144,31415926535897932,24116.10,NaN,NaN,NaN,20.003004,NaN,NaN,NaN,2.01,NaN,NaN,0,0,0,17.0
169,sinopec,2.88,NaN,NaN,NaN,22.000000,NaN,NaN,NaN,49.99,NaN,4.0,0,1,0,7.0
174,akulowe,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.00,NaN,NaN,0,1,0,7.0
179,3090,84.53,2.0,1980.5715,1378.4700,158.239230,NaN,NaN,NaN,100.00,NaN,NaN,0,0,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3490340,richardbuettner,57.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.08,NaN,NaN,0,1,0,15.0
3490371,themilliondollarhomepage,13.47,NaN,NaN,NaN,18.974686,NaN,NaN,NaN,2.49,NaN,NaN,0,1,0,24.0
3490381,metaphysical,1.36,1.0,214.5822,214.5822,110.000000,NaN,NaN,NaN,1.00,NaN,NaN,0,1,0,12.0
3490429,unicorn,29.33,NaN,NaN,NaN,77.000000,NaN,NaN,NaN,383.00,Computers,3.0,0,1,0,7.0


Мы получили два датасета, один без размеченной целевой переменной, другой без нее. Первый датасет может быть использован для обучения и проверки модели, а второй будет использоваться для отображения потенциальной цены продажи нового домена на сайте webdom.market